# Clustering — Practical Notebook

Hands-on K-Means, elbow/silhouette analysis, hierarchical clustering, DBSCAN, and customer segmentation. Synthetic data only. Run cells in order.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## 2. Generate and visualize unlabeled data

In [ ]:
X, _ = make_blobs(n_samples=480, centers=[(-4,-2),(0,4),(4,-1)],
                  cluster_std=[0.8,1.0,0.7], random_state=RANDOM_STATE)
plt.scatter(X[:,0], X[:,1], s=14, alpha=0.7)
plt.title("Unlabeled synthetic points")
plt.show()

## 3. Fit K-Means and visualize centroids

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X)
print("Inertia:", round(km.inertia_,2))
print("Cluster counts:", np.bincount(km.labels_))
plt.scatter(X[:,0], X[:,1], c=km.labels_, s=14)
plt.scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1],
            marker="X", s=180, c="black", label="Centroids")
plt.legend()
plt.show()

## 4. Elbow and silhouette scores

In [ ]:
results = []
for k in range(2,9):
    fitted = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    results.append({"k":k, "inertia":fitted.inertia_,
                    "silhouette":silhouette_score(X,fitted.labels_)})
scores = pd.DataFrame(results)
display(scores)
plt.plot(scores["k"], scores["inertia"], marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow diagnostic")
plt.show()
plt.plot(scores["k"], scores["silhouette"], marker="o")
plt.xlabel("K")
plt.ylabel("Silhouette score")
plt.title("Silhouette comparison")
plt.show()

## 5. Hierarchical agglomerative clustering

In [ ]:
hier = AgglomerativeClustering(n_clusters=3, linkage="ward")
hier_labels = hier.fit_predict(X)
plt.scatter(X[:,0], X[:,1], c=hier_labels, s=14)
plt.title("Agglomerative clustering (Ward)")
plt.show()

## 6. DBSCAN on curved shapes

In [ ]:
moons, _ = make_moons(n_samples=500, noise=0.06, random_state=RANDOM_STATE)
db = DBSCAN(eps=0.19, min_samples=6)
db_labels = db.fit_predict(moons)
cluster_count = len(set(db_labels)-{-1})
noise_count = int((db_labels==-1).sum())
print("Clusters:",cluster_count,"Noise points:",noise_count)
plt.scatter(moons[:,0],moons[:,1],c=db_labels,s=14)
plt.title("DBSCAN on two moons (-1 = noise)")
plt.show()

## 7. Compare K-Means and DBSCAN on curved data

In [ ]:
moon_km = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE).fit_predict(moons)
plt.scatter(moons[:,0],moons[:,1],c=moon_km,s=14)
plt.title("K-Means on two moons")
plt.show()
print("K-Means silhouette:",round(silhouette_score(moons,moon_km),3))
if 2 <= len(set(db_labels)) <= len(moons)-1:
    print("DBSCAN silhouette (including noise as label):",
          round(silhouette_score(moons,db_labels),3))
else:
    print("DBSCAN silhouette undefined for this labeling")

## 8. Synthetic customer segmentation

In [ ]:
n=600
latent=rng.choice(3,size=n,p=[0.35,0.40,0.25])
customers=pd.DataFrame({
    "annual_spend":np.maximum(0,rng.normal(np.array([450,1500,3200])[latent],190)),
    "visits_per_year":rng.poisson(np.array([4,10,20])[latent]),
    "avg_order_value":np.maximum(0,rng.normal(np.array([55,120,170])[latent],17)),
})
missing_rows=rng.choice(n,size=20,replace=False)
customers.loc[missing_rows,"avg_order_value"]=np.nan
display(customers.head())
display(customers.isna().sum().to_frame("missing"))

## 9. Leakage-safe preprocessing and K-Means pipeline

In [ ]:
features=["annual_spend","visits_per_year","avg_order_value"]
segmentation=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler()),
    ("kmeans",KMeans(n_clusters=3,n_init=10,random_state=RANDOM_STATE)),
])
labels=segmentation.fit_predict(customers[features])
customers["cluster"]=labels
print("Cluster sizes:")
display(customers["cluster"].value_counts().sort_index())
print("Profiles in original units:")
display(customers.groupby("cluster")[features].agg(["mean","median"]))

## 10. Internal evaluation and DBSCAN comparison

In [ ]:
Z=segmentation[:-1].transform(customers[features])
print("K-Means silhouette:",round(silhouette_score(Z,labels),3))
print("Davies-Bouldin:",round(davies_bouldin_score(Z,labels),3))
print("Calinski-Harabasz:",round(calinski_harabasz_score(Z,labels),1))
density=DBSCAN(eps=0.55,min_samples=7).fit_predict(Z)
print("DBSCAN clusters:",len(set(density)-{-1}))
print("DBSCAN noise share:",round(float(np.mean(density==-1)),3))

## 11. Interpret and document
Cluster IDs have no inherent ranking. Check segment sizes and original-unit profiles, investigate stability under different samples, and confirm proposed business actions are appropriate and fair. Do not use the latent data-generation labels as training targets.

## 12. Practice extensions
Try different K values, alternative scalers, and DBSCAN settings. Compare partitions and noise rates. Explain why an internal metric may disagree with domain usefulness.